# KNN

In [35]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


In [36]:
train_X = pd.read_csv('data/cleaned/X_train_scaled.csv', index_col=0)
train_y = pd.read_csv('data/cleaned/y_train.csv', index_col=0)['Target']
val_X = pd.read_csv('data/cleaned/X_val_scaled.csv', index_col=0)
val_y = pd.read_csv('data/cleaned/y_val.csv', index_col=0)['Target']

In [37]:
class KNNClassifier():
    def __init__(self, k):
        self.k = k

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype = np.float32)
        self.y_train = np.asarray(y).ravel()

        self._classes_ , self.y_encoded = np.unique(self.y_train, return_inverse=True)
        return self

    def predict(self, X, distance_metric, batch_size = 64):
        X = np.asarray(X, dtype = np.float32)

        if distance_metric not in ['euclidean', 'manhattan']:
            raise ValueError("Invalid distance metric. Choose 'euclidean' or 'manhattan'.")

        Y_pred = np.empty(len(X), dtype=self._classes_.dtype)
        for start in range(0, len(X), batch_size):
            batch = X[start:start + batch_size]
            #shape : (batch_size, jumlah_data_train, jumlah_fitur)
            differences = batch[:, np.newaxis, :] - self.X_train[np.newaxis, :, :]
            #cek distance metric
            if distance_metric == 'euclidean':
                distances = np.sqrt(np.sum(differences ** 2, axis=2))
            elif distance_metric == 'manhattan':
                distances = np.sum(np.abs(differences), axis=2)
            #ambil k tetangga terdekat
            nearest_indices = np.argsort(distances, axis=1)[:, :self.k]
            #ambil label dari tetangga terdekat 
            nearest_labels = self.y_encoded[nearest_indices]
            #hitung jumlah suara untuk setiap kelas
            votes = (
                nearest_labels[:, :, np.newaxis]
                == np.arange(len(self._classes_))[None, None, :]
            ).sum(axis=1)
            #ambil kelas dengan jumlah suara terbanyak
            winners = np.argmax(votes, axis=1)
            #set predicted labels untuk batch saat ini
            Y_pred[start:start + batch_size] = self._classes_[winners]

        return Y_pred

In [38]:
k = 3
metric = ['manhattan', 'euclidean']

model_diy = KNNClassifier(k)
model_diy.fit(train_X.values, train_y.to_numpy())

for m in metric:
    model_lib = KNeighborsClassifier(n_neighbors=k, metric=m)
    model_lib.fit(train_X.values, train_y.to_numpy())

    pred_diy = model_diy.predict(val_X.values, m)
    pred_lib = model_lib.predict(val_X.values)

    print(f"Accuracy on validation set our model [{m}]:", accuracy_score(val_y, pred_diy) * 100)
    print(f"Accuracy on validation set sklearn model [{m}]:", accuracy_score(val_y, pred_lib) * 100)


Accuracy on validation set our model [manhattan]: 71.29032258064515
Accuracy on validation set sklearn model [manhattan]: 71.29032258064515
Accuracy on validation set our model [euclidean]: 69.19354838709677
Accuracy on validation set sklearn model [euclidean]: 69.19354838709677


In [39]:
test_df = pd.read_csv('data/cleaned/test_df.csv', index_col=0)
student_ids = test_df['Student_ID'].to_numpy()

test_X = test_df.loc[:, train_X.columns]
target_mapping = {0: 'Dropout', 1: 'Enrolled', 2: 'Graduate'}

def decode_targets(predictions):
    labels = pd.Series(predictions).replace(target_mapping)
    if not labels.isin(target_mapping.values()).all():
        raise ValueError('Unknown target label in predictions.')
    return labels.to_numpy()

for m in metric:
    model_lib = KNeighborsClassifier(n_neighbors=k, metric=m)
    model_lib.fit(train_X.values, train_y.to_numpy())

    pred_diy = model_diy.predict(test_X.values, m)
    pred_lib = model_lib.predict(test_X.values)

    submission_diy = pd.DataFrame({'Student_ID': student_ids, 'Target': decode_targets(pred_diy)})
    submission_lib = pd.DataFrame({'Student_ID': student_ids, 'Target': decode_targets(pred_lib)})

    submission_diy.to_csv(f'./output/csv/submission_knn_{m}.csv', index=False)
    submission_lib.to_csv(f'./output/csv/submission_knn_lib_{m}.csv', index=False)


In [40]:
import joblib 

joblib.dump(model_diy, './output/knn_model_diy.pkl')

['./output/knn_model_diy.pkl']